In [12]:
import torch
import random
import math
from torch import empty
from test import *

In [13]:
# just to be sure
torch.set_grad_enabled(False)

In [14]:
# generate 2d points in [0,1] squared, targets are 0 if point inside the circle of squared radius 1/2pi and 1 outside.
# return coordinates and target tensors, both of size Nx2, plus classes tensor of size Nx1
def generate_points(nb):
    inputs = torch.empty((nb, 2))
    targets = torch.empty((nb, 2))
    classes = torch.empty((nb, 1))
    for i in range(nb):
        x = random.uniform(0,1) - 0.5
        y = random.uniform(0,1) - 0.5
        t = int(2 * math.pi * (pow(x, 2) + pow(y, 2)) < 1)
        inputs[i] = torch.tensor([x + 0.5, y + 0.5])
        targets[i,t] = 1
        targets[i,1-t] = 0
        classes[i] = t
    return inputs, targets, classes


In [15]:
# accuracy for classes prediction
def accuracy(model, inputs, classes):
    
    nb_samples = inputs.shape[0]
    pred = model.predict(inputs)
    _, pred_classes = pred.max(1)
    
    nb_errors = (pred_classes - classes[:,0]).type(torch.BoolTensor).sum().item()
    return (nb_samples - nb_errors) / nb_samples

In [16]:
# Mother class of all modules
class Module(object):
    
    # input : input to be passed into the layer
    def forward(self, *input): raise NotImplementedError
    
    # gradwrtoutput : gradient of the loss, with respect to the output of that layer (= input to next laxer)
    # return the gradient of the loss, with respect to the input of that layer (= output of previous layer)
    def backward(self, *gradwrtoutput): raise NotImplementedError
        
    def param(self): return []
    
    def to_string(self): raise NotImplementedError

In [17]:
# Sigmoid activation 
class Sigmoid(Module):
    def __init__(self):
        self.in_value = None

    def forward(self, *input):
        x = 1 / (1 + (-input[0]).exp())
        self.in_value = x
        return x
    
    def derivative(self, v):
        return self.forward(v) * self.forward(-v) # fancy form for derivative
    
    def backward(self, *gradwrtoutput):
        return self.derivative(self.in_value) * gradwrtoutput[0] # elementwise multiplication
    
    def to_string(self): return "Sigmoid"
    

In [165]:
# ReLU activation
class Relu(Module):
    def __init__(self):
        self.in_value = None

    def forward(self, *input):
        x = input[0].clone()
        x[x < 0] = 0
        self.in_value = x
        return x
    
    def derivative(self, v):
        vv = v.clone()
        vv[vv <= 0] = 0
        vv[vv > 0] = 1
        return vv
    
    def backward(self, *gradwrtoutput):
        return self.derivative(self.in_value) * gradwrtoutput[0] # elementwise multiplication
    
    def to_string(self): return "ReLU"

In [159]:
# Tanh activation
class Tanh(Module):
    def __init__(self):
        super(Tanh, self).__init__()
        self.in_value = None
    
    def forward(self, *input):
        x = ( input[0].exp() - (-input[0]).exp() ) / (input[0].exp() + (-input[0]).exp() )
        self.in_value = x
        return x
    
    def derivative(self, v):
        return 1 - self.forward(v)*self.forward(v)
    
    def backward(self, *gradwrtoutput):
        return self.derivative(self.in_value) * gradwrtoutput[0]
    
    def to_string(self): return "Tanh"

In [160]:
# MSE Loss
class MSELoss(Module):
    
    # input: predicted, target
    def forward(self, *input):
        return (input[0] - input[1]).pow(2).sum() # dividing by nb of samples ? 
    
    # gradwrtoutput : predicted, target
    def backward(self, *gradwrtoutput):
        
        return 2 * (gradwrtoutput[0] - gradwrtoutput[1]) # dividing by nb of sample ? 
    
    def to_string(self): return "MSE Loss"

In [161]:
# Linear module
class Linear(Module):
    def __init__(self, in_size, out_size):
        
        self.in_size = in_size
        self.out_size = out_size
        
        # initializing the weights
        stdv = 1. / math.sqrt(self.in_size)
        self.w = torch.empty(out_size, in_size).uniform_(-stdv, stdv)
        self.b = torch.empty(out_size).uniform_(-stdv, stdv)
        
        # initializing the gradients
        self.dw = torch.empty(out_size, in_size).zero_()
        self.db = torch.empty(out_size).zero_()
        
        self.in_value = torch.empty(in_size)
    
    # only with one input sample
    def forward(self, *input):
        self.in_value = input[0]
        return self.w.mv(input[0]) + self.b
    
    def backward(self, *gradwrtoutput):
        
        # gradient for that one sample
        dw_c = gradwrtoutput[0].view(-1, 1).mm(self.in_value.view(1, -1))
        db_c = gradwrtoutput[0]
        
        # updating stored gradients
        self.dw.add_(dw_c)
        self.db.add_(db_c)
        
        # gradient with respect to the input
        d_input = gradwrtoutput[0].view(1, -1).mm(self.w)
        
        return d_input[0]
    
    def param(self):
        return self.w, self.b, self.dw, self.db
    
    # updating the weights wrt to the step size and gradients stored, then set the gradients back to zero
    def update_param(self, step_size):
        self.w = self.w - step_size * self.dw
        self.b = self.b - step_size * self.db
        self.dw.zero_()
        self.db.zero_()
        
    def to_string(self):
        return "Linear ({}, {})".format(self.in_size, self.out_size)

In [162]:
# A sequence of modules
class Sequential(Module):
    def __init__(self, *modules, out_size):
        self.layers = modules[:-1]
        self.loss = modules[-1]
        self.out_size = out_size
        
    def forward(self, *input):
        x = input[0]
        for m in self.layers: # not forwarding through the loss
            x = m.forward(x)
        return x
    
    def train(self, inputs, targets, epochs, batch_size = 10, verbose=True):
        
        eta = 1e-1 / batch_size # divide per number of samples or not ?
                
        for e in range(epochs):
            
            sum_loss = 0
            # iterate over each sample and update weights at each iteration
            # todo : iterate over batches and update weights after each batch
            for input_batch, target_batch in zip(inputs.split(batch_size), targets.split(batch_size)):
                
                for i in range(batch_size):
                    input = input_batch[i]
                    target = target_batch[i]
                
                    # computing predicted values and loss
                    predicted = self.forward(input) 
                    sum_loss = sum_loss + self.loss.forward(predicted, target) # LOG ???

                    # computing derivative of the loss between predicted and target
                    x = self.loss.backward(predicted, target)

                    # backpropagate the loss through the net, adding the gradients in linear modules
                    for m in reversed(self.layers):
                        x = m.backward(x)
            
                # update weights in linear modules
                for m in self.layers:
                    if isinstance(m, Linear):
                        m.update_param(eta)
                        
            if verbose:        
                print("Epoch {} | Loss {:.2f}".format(e, sum_loss))
    
    def print_param(self):
        for m in self.layers:
            if isinstance(m, Linear):
                params = m.param()
                print("*** ", m.to_string(), " ***")
                print(params[0])
                print(params[1])
                print()
            
    def describe(self):
        for m in self.layers:
            print(m.to_string())
        print(self.loss.to_string())
            
    # predictions using current weights in the modules
    # input : tensor, samples aligned along first dimension
    def predict(self, inputs):
        nb_samples = inputs.shape[0]
        pred = torch.empty(nb_samples, self.out_size)
        for i in range(nb_samples):
            pred[i] = self.forward(inputs[i])
        return pred

In [169]:
def test():
    
    # model creation
    model = Sequential(
        Linear(2, 25), 
        Relu(),
        Linear(25, 25), 
        Sigmoid(),
        Linear(25, 25), 
        Tanh(),
        Linear(25, 2),
        MSELoss(), out_size = 2)
    
    print("Model:")
    print("---------------------")
    model.describe()
    print("---------------------")
    
    # generating points
    train_inputs, train_targets, train_classes = generate_points(1000)
    test_inputs, test_targets, test_classes = generate_points(1000)
    
    # train model
    model.train(train_inputs, train_targets, epochs=25, batch_size=2, verbose=True)
    
    # compute accuracy
    train_acc = accuracy(model, train_inputs, train_classes)
    test_acc = accuracy(model, test_inputs, test_classes)
    
    print("Train accuracy: {} | Test accuracy: {}".format(train_acc, test_acc))

In [170]:
test()

Model:
---------------------
Linear (2, 25)
ReLU
Linear (25, 25)
Sigmoid
Linear (25, 25)
Tanh
Linear (25, 2)
MSE Loss
---------------------
Epoch 0 | Loss 525.91
Epoch 1 | Loss 519.03
Epoch 2 | Loss 519.02
Epoch 3 | Loss 519.01
Epoch 4 | Loss 518.99
Epoch 5 | Loss 518.98
Epoch 6 | Loss 518.97
Epoch 7 | Loss 518.96
Epoch 8 | Loss 518.95
Epoch 9 | Loss 518.94
Epoch 10 | Loss 518.92
Epoch 11 | Loss 518.89
Epoch 12 | Loss 518.86
Epoch 13 | Loss 518.82
Epoch 14 | Loss 518.77
Epoch 15 | Loss 518.68
Epoch 16 | Loss 518.55
Epoch 17 | Loss 518.32
Epoch 18 | Loss 517.87
Epoch 19 | Loss 516.76
Epoch 20 | Loss 512.91
Epoch 21 | Loss 489.63
Epoch 22 | Loss 382.70
Epoch 23 | Loss 271.11
Epoch 24 | Loss 226.26
Train accuracy: 0.746 | Test accuracy: 0.742
